In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [2]:
from pathlib import Path
from data import ZarrBasinStore
store_path = Path("/scratch4/workspace/tlanghorst_umass_edu-zbs_vD")
store = ZarrBasinStore(store_path)

In [9]:
subs_df['original_comid'] = subs_df.index

# A small % of gauges are dropped if they don't match well.
common_idx = subs_df.index.intersection(gauges.index)
subs_df.loc[common_idx, 'original_comid'] = gauges.loc[common_idx, 'COMID'].astype(str)
subs_df

,is_gauge,uparea_km2,outlet_id,original_comid,COMID_v07
site_id,,,,,
23021321,False,209.123678,EAUF-A3750050,23021321,23020861
23021261,False,213.109547,EAUF-A3750050,23021261,23020837
23021244,False,291.030672,EAUF-A3750050,23021244,23020833
23021122,False,477.964586,EAUF-A3750050,23021122,23020781
23021112,False,394.295017,EAUF-A3750050,23021112,23020778
...,...,...,...,...,...
81003765,False,1613.152478,USGS-15908000,81003765,81003353
81001414,False,1473.949940,USGS-15908000,81001414,81002712
81001413,False,2751.705206,USGS-15908000,81001413,81002593


In [4]:
paths = Path("/nas/cee-ice/data/MERIT-BASINS/v07_to_bugfix").glob("v07_to_bugfix_pfaf1_*.csv")
bugfix_to_v07 = pd.concat([pd.read_csv(p, dtype=str) for p in paths]).set_index('COMID_bugfix')
subs_df = subs_df.merge(bugfix_to_v07, how='left', left_on='original_comid', right_index=True)
subs_df

,is_gauge,uparea_km2,outlet_id,original_comid,COMID_v07
site_id,,,,,
23021321,False,209.123678,EAUF-A3750050,23021321,23020861
23021261,False,213.109547,EAUF-A3750050,23021261,23020837
23021244,False,291.030672,EAUF-A3750050,23021244,23020833
23021122,False,477.964586,EAUF-A3750050,23021122,23020781
23021112,False,394.295017,EAUF-A3750050,23021112,23020778
...,...,...,...,...,...
81003765,False,1613.152478,USGS-15908000,81003765,81003353
81001414,False,1473.949940,USGS-15908000,81001414,81002712
81001413,False,2751.705206,USGS-15908000,81001413,81002593


In [5]:
MHS_dir = Path("/nas/cee-water/cjgleason/heejin/HYDRO_combination/02Result/Unified_Hydrography/")

mhs = pd.concat([pd.read_csv(f)[['COMID','reach_id']].drop_duplicates().dropna() for f in MHS_dir.glob("all_*_MHS.csv")])
mhs

,COMID,reach_id
320,13037099,1.327510e+10
323,13037099,1.327510e+10
326,13037099,1.327510e+10
355,13066945,1.322620e+10
386,13106853,1.312200e+10
...,...,...
2327271,91007672,9.141000e+10
2327278,91007672,9.141000e+10
2327306,91007671,9.141000e+10
2327307,91007671,9.141000e+10


In [6]:
subs_df.dropna(subset='COMID_v07', inplace=True)
subs_df['COMID_v07'] = subs_df['COMID_v07'].astype(int)

In [7]:
merged = subs_df.reset_index().merge(mhs, left_on='COMID_v07', right_on='COMID', how='left')
merged.dropna(subset='reach_id', inplace=True)
merged = merged.set_index('site_id')
merged

,is_gauge,uparea_km2,outlet_id,original_comid,COMID_v07,COMID,reach_id
site_id,,,,,,,
23020936,False,625.114709,EAUF-A3750050,23020936,23020606,23020606.0,2.326900e+10
23020936,False,625.114709,EAUF-A3750050,23020936,23020606,23020606.0,2.326900e+10
23020927,False,948.431597,EAUF-A3750050,23020927,23020605,23020605.0,2.326900e+10
23020916,False,952.117079,EAUF-A3750050,23020916,23020713,23020713.0,2.326800e+10
23020913,False,775.731383,EAUF-A3750050,23020913,23020712,23020712.0,2.326900e+10
...,...,...,...,...,...,...,...
81001413,False,2751.705206,USGS-15908000,81001413,81002593,81002593.0,8.139040e+10
81001413,False,2751.705206,USGS-15908000,81001413,81002593,81002593.0,8.139040e+10
81001413,False,2751.705206,USGS-15908000,81001413,81002593,81002593.0,8.139040e+10


In [11]:
merged[merged['is_gauge']].to_parquet(metadata_dir / 'gauge_matchups.parquet')

In [8]:
def get_h_w_outlier_mask(swot: pd.DataFrame, threshold: int):
    def calc_mad_z(group):
        median = group.median()
        mad = (group - median).abs().median()
        
        if mad == 0:
            return pd.Series(0, index=group.index)
            
        modified_z = 0.6745 * (group - median) / mad
        return modified_z 
    
    w_z = swot.groupby('reach_id')['width'].transform(calc_mad_z)
    h_z = swot.groupby('reach_id')['wse'].transform(calc_mad_z)

    # individual 
    w_outlier = w_z.abs() > threshold
    h_outlier = h_z.abs() > threshold
    
    # 1. Outlier in only one metric (xor)
    mask_xor = w_outlier ^ h_outlier
    
    # 2. Outliers in both, but in opposite directions 
    mask_opposite = w_outlier & h_outlier & (w_z * h_z < 0)
    
    return mask_xor | mask_opposite


In [ ]:
import pyarrow.dataset as ds
import itertools 

fields = [
    'reach_id', 'datetime',
    'wse', 'wse_u', 'wse_r_u', 
    'slope','slope_u','slope_r_u',
    'slope2','slope2_u','slope2_r_u',
    'width','width_u',
    'area_total','area_tot_u','area_detct','area_det_u','area_wse',
    'layovr_val','node_dist','loc_offset','xtrk_dist','reach_q','reach_q_b',
    'dark_frac','ice_clim_f','ice_dyn_f','partial_f','obs_frac_n',
]

reaches = merged['reach_id'].astype(int).astype(str).values
filters = [('reach_id', 'in', reaches), ('wse','!=',-999999999999.0)]


# version D
reach_dir = Path("/nas/cee-ice/data/SWOT/RiverSP_Reach_vD/parquet")
files = list(reach_dir.glob('*.parquet'))
swot_dfs = [pd.read_parquet(f, columns=fields, filters=filters) for f in files]
    
swot = pd.concat(swot_dfs)
swot = swot[swot['reach_q_b'] < 2**5].copy()
swot = swot[swot['wse_u'] < 0.20]

outlier_mask = get_h_w_outlier_mask(swot, 5)
swot = swot[~outlier_mask]

swot['d_wse'] = swot['wse'] - swot.groupby('reach_id')['wse'].transform('median')
swot['d_width'] = swot['width'] - swot.groupby('reach_id')['width'].transform('median')
swot['d_area'] = swot['d_wse'] * (swot['width'] + swot.groupby('reach_id')['width'].transform('median'))/2

swot = swot[swot['d_wse'].abs() < 10] # > 10 m from median height is very likely wrong

swot = swot.set_index(['reach_id','datetime'])

In [ ]:
swot.xs('81110200041', level='reach_id').plot.scatter('wse','width')

In [ ]:
def get_best_record(grp):
    return grp.sort_values('wse_u').iloc[0]


def safe_xs(df, val, level):
    try:
        return df.xs(val, level=level)
    except KeyError:
        return pd.DataFrame()
        
def get_swot_r(reach_id, swot):
    swot_reach = pd.concat([safe_xs(swot, rid, 'reach_id') for rid in reach_ids])
        
    swot_reach.index = pd.to_datetime(swot_reach.index).round('D').tz_localize('UTC')
    swot_reach.index.name = 'date'
    
    # For any duplicated dates, pick the best obs according to uncertainty estimate.
    if any(swot_reach.index.duplicated()):
        swot_reach = swot_reach.groupby(swot_reach.index.name).apply(get_best_record)

    return swot_reach.add_suffix('_river')


BATCH_SIZE = 1000
batches = []
for basin_id, basin_group in merged.groupby('outlet_id'):
    basin_subs = basin_subbasin_dict[basin_id]
    n_subs = len(basin_subs)

    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        batches.append((basin_id, batch_subs))


for basin_id, batch_subs in tqdm(batches):
    basin_subs = basin_subbasin_dict[basin_id]
    batch_df_list = []
    for subbasin in batch_subs:
        try:
            reach_ids = merged.loc[subbasin]['reach_id']
        except:
            continue

        if isinstance(reach_ids, pd.Series):
            reach_ids = reach_ids.values.astype(int).astype(str)
        elif isinstance(reach_ids, np.float64):
            reach_ids = [str(int(reach_ids))]
            
        swot_df = get_swot_r(reach_ids, swot)

        if swot_df is not None and not swot_df.empty:
            swot_df['subbasin'] = subbasin
            swot_df.index.name = 'date'
            batch_df_list.append(swot_df)

    # If the whole batch is empty, we technically don't need to write anything 
    # since the zarr was initialized with NaNs/empty).
    if batch_df_list:
        batch_df = pd.concat(batch_df_list)
        store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs)

In [ ]:
swot_vars = [
    'wse_river',
     'wse_u_river',
     'wse_r_u_river',
     'slope_river',
     'slope_u_river',
     'slope_r_u_river',
     'slope2_river',
     'slope2_u_river',
     'slope2_r_u_river',
     'width_river',
     'width_u_river',
     'area_total_river',
     'area_tot_u_river',
     'area_detct_river',
     'area_det_u_river',
     'area_wse_river',
     'layovr_val_river',
     'node_dist_river',
     'loc_offset_river',
     'xtrk_dist_river',
     'reach_q_river',
     'reach_q_b_river',
     'dark_frac_river',
     'ice_clim_f_river',
     'ice_dyn_f_river',
     'partial_f_river',
     'obs_frac_n_river',
     'd_wse_river',
     'd_width_river',
     'd_area_river',
     'subbasin'
]

store.compute_and_store_stats(
    var_names = swot_vars,
    overwrite = True,
    n_workers= 8,
    memory_limit = '4GB'
)

Dask Dashboard: http://127.0.0.1:8787/status


Processing Basins:  92%|█████████▏| 1352/1472 [3:24:17<08:50,  4.42s/it]  

In [ ]:
any(swot_df.index.duplicated())

In [ ]:
swot.xs('111', level='reach_id')

In [ ]:
merged.set_index('site_id')